# Memory

<div style="display: flex; justify-content: flex-start; gap: 10px;">
  <img src="./assets/LC_Memory_before.png" style="width:500px; border:1px solid #ccc; border-radius:6px;">
</div>

Persisting messages, or 'agent state' between invocations of the agent.

## Setup

Load and/or check for needed environmental variables

In [1]:
from dotenv import load_dotenv
from env_utils import doublecheck_env

# Load environment variables from .env
load_dotenv()

# Check and print results
doublecheck_env("example.env")

CRUSOE_API_KEY=****bmsK
LANGSMITH_API_KEY=****here
LANGSMITH_TRACING=false
LANGSMITH_PROJECT=****ials


In [2]:
from sql_db import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///Chinook.db")

In [3]:
from dataclasses import dataclass

@dataclass
class RuntimeContext:
    db: SQLDatabase

In [4]:
from langchain_core.tools import tool
from langgraph.runtime import get_runtime

@tool
def execute_sql(query: str) -> str:
    """Execute a SQLite command and return results."""
    runtime = get_runtime(RuntimeContext)
    db = runtime.context.db

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

In [5]:
SYSTEM_PROMPT = """You are a careful SQLite analyst.

Rules:
- Think step-by-step.
- When you need data, call the tool `execute_sql` with ONE SELECT query.
- Read-only only; no INSERT/UPDATE/DELETE/ALTER/DROP/CREATE/REPLACE/TRUNCATE.
- Limit to 5 rows unless the user explicitly asks otherwise.
- If the tool returns 'Error:', revise the SQL and try again.
- Prefer explicit column lists; avoid SELECT *.
"""

In [6]:
from langchain_crusoe import ChatCrusoe
from langchain.agents import create_agent

agent = create_agent(
    model=ChatCrusoe(model="zai/GLM-5.2"),
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
)

## Repeated Queries

In [7]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    stream_mode="values",
    context=RuntimeContext(db=db),
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?


================================== Ai Message ==================================
Tool Calls:
  execute_sql (chatcmpl-tool-814b9339d958c766)
 Call ID: chatcmpl-tool-814b9339d958c766
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table'
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',), ('Invoice',), ('InvoiceLine',), ('MediaType',), ('Playlist',), ('PlaylistTrack',), ('Track',)]


================================== Ai Message ==================================
Tool Calls:
  execute_sql (chatcmpl-tool-a5933fc6700dd3dd)
 Call ID: chatcmpl-tool-a5933fc6700dd3dd
  Args:
    query: SELECT c.CustomerId, c.FirstName, c.LastName 
FROM Customer c 
WHERE c.FirstName = 'Frank' AND c.LastName = 'Harris'
================================= Tool Message =================================
Name: execute_sql

[(16, 'Frank', 'Harris')]


================================== Ai Message ==================================
Tool Calls:
  execute_sql (chatcmpl-tool-8260698a45171d37)
 Call ID: chatcmpl-tool-8260698a45171d37
  Args:
    query: SELECT InvoiceId, InvoiceDate, Total 
FROM Invoice 
WHERE CustomerId = 16 
ORDER BY InvoiceDate DESC 
LIMIT 1
================================= Tool Message =================================
Name: execute_sql

[(374, '2013-07-04 00:00:00', 5.94)]


================================== Ai Message ==================================

Hi Frank! Your most recent invoice was **Invoice #374**, dated **July 4, 2013**, with a total of **$5.94**.


In [8]:
question = "What were the titles?"

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What were the titles?


================================== Ai Message ==================================

I'd be happy to help you find titles! Let me first explore the database to understand its structure.
Tool Calls:
  execute_sql (chatcmpl-tool-b3dc9eb95f603b35)
 Call ID: chatcmpl-tool-b3dc9eb95f603b35
  Args:
    query: SELECT name, type FROM sqlite_master WHERE type IN ('table', 'view')
================================= Tool Message =================================
Name: execute_sql

[('Album', 'table'), ('Artist', 'table'), ('Customer', 'table'), ('Employee', 'table'), ('Genre', 'table'), ('Invoice', 'table'), ('InvoiceLine', 'table'), ('MediaType', 'table'), ('Playlist', 'table'), ('PlaylistTrack', 'table'), ('Track', 'table')]


================================== Ai Message ==================================

I can see the database has several tables. There are a few possibilities for what "titles" could refer to — such as **Album titles**, **Track names**, or **Employee job titles**. Let me check the most likely candidates:
Tool Calls:
  execute_sql (chatcmpl-tool-8ecc01105bfc8247)
 Call ID: chatcmpl-tool-8ecc01105bfc8247
  Args:
    query: PRAGMA table_info(Album)
  execute_sql (chatcmpl-tool-9d6d835e7744dd41)
 Call ID: chatcmpl-tool-9d6d835e7744dd41
  Args:
    query: PRAGMA table_info(Track)
  execute_sql (chatcmpl-tool-bc916ab7e874eae0)
 Call ID: chatcmpl-tool-bc916ab7e874eae0
  Args:
    query: PRAGMA table_info(Employee)
================================= Tool Message =================================
Name: execute_sql

[(0, 'EmployeeId', 'INTEGER', 1, None, 1), (1, 'LastName', 'NVARCHAR(20)', 1, None, 0), (2, 'FirstName', 'NVARCHAR(20)', 1, None, 0), (3, 'Title', 'NVARCHAR(30)', 0, None, 0), (4, 'Report

================================== Ai Message ==================================

I found that both the **Album** table and the **Employee** table have a `Title` column. Let me fetch titles from both:
Tool Calls:
  execute_sql (chatcmpl-tool-9264710cb177515c)
 Call ID: chatcmpl-tool-9264710cb177515c
  Args:
    query: SELECT Title FROM Album LIMIT 5
  execute_sql (chatcmpl-tool-a370c3c72fe69d62)
 Call ID: chatcmpl-tool-a370c3c72fe69d62
  Args:
    query: SELECT FirstName, LastName, Title FROM Employee LIMIT 5
================================= Tool Message =================================
Name: execute_sql

[('Andrew', 'Adams', 'General Manager'), ('Nancy', 'Edwards', 'Sales Manager'), ('Jane', 'Peacock', 'Sales Support Agent'), ('Margaret', 'Park', 'Sales Support Agent'), ('Steve', 'Johnson', 'Sales Support Agent')]


================================== Ai Message ==================================

Here's what I found for "titles" in the database:

### 📀 Album Titles
| # | Title |
|---|-------|
| 1 | For Those About To Rock We Salute You |
| 2 | Balls to the Wall |
| 3 | Restless and Wild |
| 4 | Let There Be Rock |
| 5 | Big Ones |

### 👤 Employee Job Titles
| Name | Title |
|------|-------|
| Andrew Adams | General Manager |
| Nancy Edwards | Sales Manager |
| Jane Peacock | Sales Support Agent |
| Margaret Park | Sales Support Agent |
| Steve Johnson | Sales Support Agent |

Would you like more details on either of these, or were you referring to something else (like track names)? Let me know!


## Add memory

In [9]:
from langgraph.checkpoint.memory import InMemorySaver

In [10]:
from langchain_crusoe import ChatCrusoe
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage

agent = create_agent(
    model=ChatCrusoe(model="zai/GLM-5.2"),
    tools=[execute_sql],
    system_prompt=SYSTEM_PROMPT,
    context_schema=RuntimeContext,
    checkpointer=InMemorySaver(),
)

In [11]:
question = "This is Frank Harris, What was the total on my last invoice?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

This is Frank Harris, What was the total on my last invoice?


================================== Ai Message ==================================

I'd be happy to help you find the total on your last invoice, Frank. Let me start by looking at the database structure.
Tool Calls:
  execute_sql (chatcmpl-tool-8cb579a40f30aa68)
 Call ID: chatcmpl-tool-8cb579a40f30aa68
  Args:
    query: SELECT name FROM sqlite_master WHERE type='table'
================================= Tool Message =================================
Name: execute_sql

[('Album',), ('Artist',), ('Customer',), ('Employee',), ('Genre',), ('Invoice',), ('InvoiceLine',), ('MediaType',), ('Playlist',), ('PlaylistTrack',), ('Track',)]


================================== Ai Message ==================================
Tool Calls:
  execute_sql (chatcmpl-tool-a51741f162012623)
 Call ID: chatcmpl-tool-a51741f162012623
  Args:
    query: SELECT sql FROM sqlite_master WHERE type='table' AND name IN ('Invoice', 'Customer')
================================= Tool Message =================================
Name: execute_sql

[('CREATE TABLE [Customer]\n(\n    [CustomerId] INTEGER  NOT NULL,\n    [FirstName] NVARCHAR(40)  NOT NULL,\n    [LastName] NVARCHAR(20)  NOT NULL,\n    [Company] NVARCHAR(80),\n    [Address] NVARCHAR(70),\n    [City] NVARCHAR(40),\n    [State] NVARCHAR(40),\n    [Country] NVARCHAR(40),\n    [PostalCode] NVAR...',), ('CREATE TABLE [Invoice]\n(\n    [InvoiceId] INTEGER  NOT NULL,\n    [CustomerId] INTEGER  NOT NULL,\n    [InvoiceDate] DATETIME  NOT NULL,\n    [BillingAddress] NVARCHAR(70),\n    [BillingCity] NVARCHAR(40),\n    [BillingState] NVARCHAR(40),\n    [BillingCountry] NVARCHAR(40),\n    [BillingPosta

================================== Ai Message ==================================
Tool Calls:
  execute_sql (chatcmpl-tool-a3678cb9c003c55f)
 Call ID: chatcmpl-tool-a3678cb9c003c55f
  Args:
    query: SELECT InvoiceId, InvoiceDate, Total FROM Invoice WHERE CustomerId = (SELECT CustomerId FROM Customer WHERE FirstName = 'Frank' AND LastName = 'Harris') ORDER BY InvoiceDate DESC LIMIT 1
================================= Tool Message =================================
Name: execute_sql

[(374, '2013-07-04 00:00:00', 5.94)]


================================== Ai Message ==================================

Hi Frank! Here are the details of your last invoice:

- **Invoice ID:** 374
- **Date:** July 4, 2013
- **Total:** **$5.94**

Is there anything else you'd like to know about this invoice or any other purchases?


In [12]:
question = "What were the titles?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

What were the titles?


================================== Ai Message ==================================
Tool Calls:
  execute_sql (chatcmpl-tool-b8d56c94cfaa5feb)
 Call ID: chatcmpl-tool-b8d56c94cfaa5feb
  Args:
    query: SELECT t.Name AS TrackTitle, a.Title AS AlbumTitle FROM InvoiceLine il JOIN Track t ON il.TrackId = t.TrackId JOIN Album a ON t.AlbumId = a.AlbumId WHERE il.InvoiceId = 374
================================= Tool Message =================================
Name: execute_sql

[('Holier Than Thou', 'Black Album'), ('Through The Never', 'Black Album'), ('My Friend Of Misery', 'Black Album'), ('The Wait', 'Garage Inc. (Disc 2)'), ('Blitzkrieg', 'Garage Inc. (Disc 2)'), ('So What', 'Garage Inc. (Disc 2)')]


================================== Ai Message ==================================

Here are the track titles from your last invoice:

| # | Track Title | Album |
|---|-------------|-------|
| 1 | Holier Than Thou | Black Album |
| 2 | Through The Never | Black Album |
| 3 | My Friend Of Misery | Black Album |
| 4 | The Wait | Garage Inc. (Disc 2) |
| 5 | Blitzkrieg | Garage Inc. (Disc 2) |
| 6 | So What | Garage Inc. (Disc 2) |

Looks like you had a nice mix of Metallica tracks — three from the *Black Album* and three from *Garage Inc. (Disc 2)*. Is there anything else you'd like to know?


## Try your own queries
Now that there is memory, check the agents recall!

In [13]:
question = "Your Question Here?"
steps = []

for step in agent.stream(
    {"messages": [{"role": "user", "content": question}]},
    {"configurable": {"thread_id": "1"}},
    context=RuntimeContext(db=db),
    stream_mode="values",
):
    step["messages"][-1].pretty_print()
    steps.append(step)

================================ Human Message =================================

Your Question Here?


================================== Ai Message ==================================

It looks like that message might have been sent by mistake! Is there anything else you'd like to know about your invoices or purchases, Frank? I'm happy to help!
